In [1]:
# ==========================================
# INSTALL — Qwen3-8B LoRA deps + prebuilt FlashAttention wheel
# Colab A100 default: Python 3.12 + torch 2.10.0+cu128
# ==========================================

import sys
import subprocess

INSTALLATION_OK = False

def run(cmd):
    print(f"\n$ {cmd}", flush=True)
    result = subprocess.run(cmd, shell=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(f"Command failed: {cmd}")

# Keep Colab's existing torch. Do NOT reinstall torch.
run(f"""{sys.executable} -m pip install -U \
"transformers>=4.51.0" \
"sentence-transformers>=5.0.0" \
"datasets>=2.19.0" \
"accelerate>=0.30.0" \
"peft>=0.12.0" \
safetensors tqdm scikit-learn packaging ninja""")

# Fix PEFT/torchao compatibility for torch 2.10.0+cu128
run(f"{sys.executable} -m pip install -U torchao --index-url https://download.pytorch.org/whl/cu128")

# Prebuilt community wheel for:
# flash-attn 2.8.3 + CUDA 12 + torch 2.10 + Python 3.12 + Linux x86_64
FLASH_ATTN_WHEEL = "https://github.com/lesj0610/flash-attention/releases/download/v2.8.3-cu12-torch2.10-cp312/flash_attn-2.8.3%2Bcu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl"

run(f'{sys.executable} -m pip install -U "{FLASH_ATTN_WHEEL}"')

# Verification
import torch
import transformers
import flash_attn
from transformers.utils import is_flash_attn_2_available

print("\n===== VERIFICATION =====")
print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA used by torch:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)
print("BF16 supported:", torch.cuda.is_bf16_supported() if torch.cuda.is_available() else None)
print("transformers:", transformers.__version__)
print("flash_attn:", getattr(flash_attn, "__version__", "unknown"))
print("FA2 available to Transformers:", is_flash_attn_2_available())

assert torch.cuda.is_available(), "CUDA is not available"
assert torch.cuda.is_bf16_supported(), "BF16 is not supported"
assert is_flash_attn_2_available(), "FlashAttention 2 is not available to Transformers"

INSTALLATION_OK = True
print("\n✅ INSTALLATION_OK = True")


$ /usr/bin/python3 -m pip install -U "transformers>=4.51.0" "sentence-transformers>=5.0.0" "datasets>=2.19.0" "accelerate>=0.30.0" "peft>=0.12.0" safetensors tqdm scikit-learn packaging ninja

$ /usr/bin/python3 -m pip install -U torchao --index-url https://download.pytorch.org/whl/cu128

$ /usr/bin/python3 -m pip install -U "https://github.com/lesj0610/flash-attention/releases/download/v2.8.3-cu12-torch2.10-cp312/flash_attn-2.8.3%2Bcu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl"

===== VERIFICATION =====
torch: 2.10.0+cu128
CUDA available: True
CUDA used by torch: 12.8
GPU: NVIDIA A100-SXM4-40GB
BF16 supported: True
transformers: 5.7.0
flash_attn: 2.8.3
FA2 available to Transformers: True

✅ INSTALLATION_OK = True


In [2]:
assert INSTALLATION_OK is True, "Installation failed. Do not start training."

# ==========================================
# 1. IMPORTS + CHECKS
# ==========================================
import os
import gc
import json
import gzip
import math
import random
import shutil
from typing import Any, Dict, List, Tuple

import numpy as np
import torch
import torchao

import transformers
from transformers import EarlyStoppingCallback
from transformers.utils import is_flash_attn_2_available

from datasets import load_dataset, Dataset
from tqdm.auto import tqdm

from peft import LoraConfig, TaskType

from sentence_transformers import (
    SentenceTransformer,
    SentenceTransformerTrainer,
    SentenceTransformerTrainingArguments,
    losses,
)

try:
    from sentence_transformers.training_args import BatchSamplers
except Exception:
    from sentence_transformers.base.sampler import BatchSamplers

try:
    from sentence_transformers.evaluation import SentenceEvaluator
except Exception:
    from sentence_transformers.base.evaluation import BaseEvaluator as SentenceEvaluator

print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("CUDA available:", torch.cuda.is_available())
print("torchao:", torchao.__version__)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("BF16 supported:", torch.cuda.is_bf16_supported())

assert torch.cuda.is_available(), "CUDA GPU is required."
assert is_flash_attn_2_available(), "FlashAttention 2 is not available."

torch: 2.10.0+cu128
transformers: 5.7.0
CUDA available: True
torchao: 0.17.0+cu128
GPU: NVIDIA A100-SXM4-40GB
BF16 supported: True


/tmp/ipykernel_7097/1303168761.py:28: DeprecationWarning: Importing from 'sentence_transformers.losses' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.losses' instead.
  from sentence_transformers import (
/tmp/ipykernel_7097/1303168761.py:36: DeprecationWarning: Importing from 'sentence_transformers.training_args' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.training_args' instead.
  from sentence_transformers.training_args import BatchSamplers
/tmp/ipykernel_7097/1303168761.py:41: DeprecationWarning: Importing from 'sentence_transformers.evaluation' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.evaluation' instead.
  from sentence_transformers.evaluation import SentenceEvaluator


In [3]:
# ==========================================
# CT26 TASK 1 — QWEN3-EMBEDDING-8B LORA FINETUNING
# Fair comparison to E5 retriever:
# - Same train/dev splits
# - Same core loss family: Multiple Negatives Ranking Loss
# - Same effective batch size: 128
# - Same max epochs: 15
# - Same LR: 1e-5
# - Same early stopping patience: 3
# - Same best checkpoint selection by multilingual avg MRR@5
#
# Difference:
# - Uses LoRA adapters instead of full finetuning
# - Uses CachedMultipleNegativesRankingLoss so effective batch=128 is real
# - Uses FlashAttention 2, required
# ==========================================

# ==========================================
# 2. GOOGLE DRIVE + PATHS
# ==========================================
USE_GOOGLE_DRIVE = True

if USE_GOOGLE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        DRIVE_ROOT = "/content/drive/MyDrive"
    except Exception as e:
        print("Drive mount failed, using local path:", repr(e))
        DRIVE_ROOT = "."
else:
    DRIVE_ROOT = "."

ROOT_DIR = os.path.join(DRIVE_ROOT, "ct26_qwen3_embedding_8b_lora")

from datetime import datetime

RESET_EXISTING_ROOT = True
BACKUP_EXISTING_ROOT = False

if RESET_EXISTING_ROOT and os.path.isdir(ROOT_DIR):
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

    if BACKUP_EXISTING_ROOT:
        BACKUP_DIR = ROOT_DIR + f"_BACKUP_{timestamp}"
        print(f"Existing ROOT_DIR found. Moving it to backup:\n{BACKUP_DIR}")
        shutil.move(ROOT_DIR, BACKUP_DIR)
    else:
        print(f"Existing ROOT_DIR found. Deleting it:\n{ROOT_DIR}")
        shutil.rmtree(ROOT_DIR)

CHECKPOINT_DIR = os.path.join(ROOT_DIR, "trainer_checkpoints")
BEST_MODEL_DIR = os.path.join(ROOT_DIR, "best_qwen3_8b_lora_sentence_transformer")
ARTIFACTS_DIR = os.path.join(ROOT_DIR, "retrieval_artifacts")
EVAL_DIR = os.path.join(ROOT_DIR, "eval")

os.makedirs(ROOT_DIR, exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(BEST_MODEL_DIR, exist_ok=True)
os.makedirs(ARTIFACTS_DIR, exist_ok=True)
os.makedirs(EVAL_DIR, exist_ok=True)

print("ROOT_DIR:", ROOT_DIR)

# ==========================================
# 3. CONFIG
# ==========================================
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DATASET_NAME = "sschellhammer/CT26_Task1_SourceRetrievalForScientificWebClaims"
LANGUAGES = ["en", "fr", "de"]
MODEL_NAME = "Qwen/Qwen3-Embedding-8B"

HF_TOKEN = os.environ.get("HF_TOKEN", None)

# Keep close to E5 experiment
EFFECTIVE_BATCH_SIZE = 128
MAX_EPOCHS = 3
LEARNING_RATE = 1e-5
EARLY_STOPPING_PATIENCE = 3

# Qwen-specific practical settings
MAX_SEQ_LENGTH = 512
EVAL_DOC_BATCH_SIZE = 16
EVAL_QUERY_BATCH_SIZE = 32

# CachedMNRL internal forward mini-batch.
# On A100 40GB try 4. If OOM, reduce to 2 or 1.
CACHED_LOSS_MINI_BATCH_SIZE = 4

# LoRA config.
# r=32 gives meaningful capacity without going crazy.
# target_modules includes attention + MLP projections for stronger adaptation.
LORA_R = 32
LORA_ALPHA = 64
LORA_DROPOUT = 0.05
LORA_TARGET_MODULES = [
    "q_proj", "k_proj", "v_proj", "o_proj",
    "gate_proj", "up_proj", "down_proj",
]

DELETE_TRAINER_CHECKPOINTS_AFTER_EXPORT = False
SAVE_FINAL_DEV_CANDIDATES = True
SAVE_CANDIDATE_TOPK = 100

# This was your best Qwen zero-shot instruction.
TASK_INSTRUCTION = (
    "Given a scientific web claim, retrieve the title and abstract of the "
    "scientific publication that is the source or best evidence for the claim."
)

# Your current best E5 dev baseline
E5_BASELINE = {
    "MRR@1": 0.5548,
    "MRR@5": 0.6282,
    "MRR@10": 0.6350,
    "Recall@5": 0.7416,
    "Recall@10": 0.7917,
}

# ==========================================
# 4. HELPERS
# ==========================================
def cleanup_cuda():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def normalize_ws(text: Any) -> str:
    return " ".join(str(text).split())

def get_detailed_instruct(task_description: str, query: str) -> str:
    return f"Instruct: {task_description}\nQuery:{query}"

def save_json(path: str, obj: Any):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)

def save_json_gz(path: str, obj: Any):
    with gzip.open(path, "wt", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False)

def dataset_kwargs():
    kwargs = {}
    if HF_TOKEN:
        kwargs["token"] = HF_TOKEN
    return kwargs

def compute_metrics_from_ranks(ranks: List[int]) -> Dict[str, float]:
    ranks = np.asarray(ranks)

    return {
        "MRR@1": float(np.mean([1.0 / r if r <= 1 else 0.0 for r in ranks])),
        "MRR@5": float(np.mean([1.0 / r if r <= 5 else 0.0 for r in ranks])),
        "MRR@10": float(np.mean([1.0 / r if r <= 10 else 0.0 for r in ranks])),
        "MRR@50": float(np.mean([1.0 / r if r <= 50 else 0.0 for r in ranks])),
        "MRR@100": float(np.mean([1.0 / r if r <= 100 else 0.0 for r in ranks])),

        "Recall@1": float((ranks <= 1).mean()),
        "Recall@5": float((ranks <= 5).mean()),
        "Recall@10": float((ranks <= 10).mean()),
        "Recall@50": float((ranks <= 50).mean()),
        "Recall@100": float((ranks <= 100).mean()),
    }

def compute_metrics(q_emb: np.ndarray, doc_emb: np.ndarray, gt_ids: List[str], docid_to_idx: Dict[str, int]):
    scores = q_emb @ doc_emb.T

    ranks = []
    for i, gold_id in enumerate(gt_ids):
        gt_idx = docid_to_idx.get(str(gold_id), None)

        if gt_idx is None:
            ranks.append(10000)
            continue

        gold_score = scores[i, gt_idx]
        rank = int(np.sum(scores[i] > gold_score)) + 1
        ranks.append(rank)

    return compute_metrics_from_ranks(ranks), ranks

def get_topk_candidates(q_emb: np.ndarray, doc_emb: np.ndarray, doc_ids_list: List[str], topk: int):
    scores = q_emb @ doc_emb.T
    topk = min(topk, scores.shape[1])

    part = np.argpartition(-scores, kth=topk - 1, axis=1)[:, :topk]
    part_scores = np.take_along_axis(scores, part, axis=1)
    order = np.argsort(-part_scores, axis=1)

    top_idx = np.take_along_axis(part, order, axis=1)
    top_scores = np.take_along_axis(part_scores, order, axis=1)

    top_ids = [[doc_ids_list[j] for j in row] for row in top_idx]
    top_scores = top_scores.astype(np.float32).tolist()

    return top_ids, top_scores

def print_metrics(title: str, metrics: Dict[str, Any]):
    print(f"\n===== {title} =====")

    metric_order = [
        "MRR@1", "MRR@5", "MRR@10", "MRR@50", "MRR@100",
        "Recall@1", "Recall@5", "Recall@10", "Recall@50", "Recall@100",
    ]

    for lang in LANGUAGES:
        print(f"--- {lang.upper()} ---")
        for key in metric_order:
            print(f"  {key:10s}: {metrics['per_language'][lang][key]:.4f}")

    print("--- MULTILINGUAL AVG ---")
    for key in metric_order:
        print(f"  {key:10s}: {metrics['average'][key]:.4f}")

def compare_to_e5(metrics: Dict[str, Any], title: str):
    print(f"\n===== COMPARISON TO E5 BASELINE | {title} =====")
    avg = metrics["average"]

    for key in ["MRR@1", "MRR@5", "MRR@10", "Recall@5", "Recall@10"]:
        q_val = avg[key]
        e_val = E5_BASELINE[key]
        delta = q_val - e_val
        sign = "+" if delta >= 0 else ""
        print(f"{key:10s} QwenLoRA={q_val:.4f} | E5={e_val:.4f} | Δ={sign}{delta:.4f}")

# ==========================================
# 5. LOAD COLLECTION
# ==========================================
print("\nLoading scientific collection...")
collection_data = load_dataset(DATASET_NAME, "collection", **dataset_kwargs())["collection"]

doc_dict = {}
doc_ids_list = []
doc_texts_list = []
doc_raw = {}

print("Preparing collection...")
for item in tqdm(collection_data, desc="Collection"):
    doc_id = str(item["pubkey"])

    title = normalize_ws(item.get("title", ""))
    abstract = normalize_ws(item.get("abstract", ""))

    # Qwen docs: no document-side instruction.
    retrieval_text = normalize_ws(f"Title: {title} Abstract: {abstract}")
    raw_text = retrieval_text

    doc_dict[doc_id] = retrieval_text
    doc_ids_list.append(doc_id)
    doc_texts_list.append(retrieval_text)
    doc_raw[doc_id] = raw_text

docid_to_idx = {did: idx for idx, did in enumerate(doc_ids_list)}

print(f"Documents loaded: {len(doc_ids_list)}")

save_json(os.path.join(ARTIFACTS_DIR, "doc_ids_list.json"), doc_ids_list)
save_json_gz(os.path.join(ARTIFACTS_DIR, "doc_raw.json.gz"), doc_raw)

# ==========================================
# 6. LOAD TRAIN + DEV DATA
# ==========================================
print("\nLoading train/dev data...")

combined_train_data = []
dev_by_lang = {}

for lang in LANGUAGES:
    train_split = load_dataset(DATASET_NAME, lang, split="train", **dataset_kwargs())
    dev_split = load_dataset(DATASET_NAME, lang, split="dev", **dataset_kwargs())

    for item in train_split:
        row = dict(item)
        row["lang"] = lang
        combined_train_data.append(row)

    dev_queries = []
    dev_labels = []

    for item in dev_split:
        q = normalize_ws(item["text"])
        dev_queries.append(get_detailed_instruct(TASK_INSTRUCTION, q))
        dev_labels.append(str(item["pubkey"]))

    dev_by_lang[lang] = {
        "queries": dev_queries,
        "labels": dev_labels,
        "raw_rows": [dict(x) for x in dev_split],
    }

    print(f"{lang}: train={len(train_split)}, dev={len(dev_split)}")

print("Total train items:", len(combined_train_data))

# ==========================================
# 7. BUILD TRAIN DATASET
# ==========================================
print("\nBuilding Qwen LoRA training pairs...")

anchors = []
positives = []
skipped = 0

for item in tqdm(combined_train_data, desc="Train pairs"):
    claim = normalize_ws(item["text"])
    gold_id = str(item["pubkey"])

    if gold_id not in doc_dict:
        skipped += 1
        continue

    # Query gets instruction.
    anchors.append(get_detailed_instruct(TASK_INSTRUCTION, claim))

    # Document does not get instruction.
    positives.append(doc_dict[gold_id])

train_dataset = Dataset.from_dict({
    "anchor": anchors,
    "positive": positives,
})

print("Training dataset size:", len(train_dataset))
print("Skipped:", skipped)

# ==========================================
# 8. CUSTOM DEV EVALUATOR
# ==========================================
class CT26QwenDevEvaluator(SentenceEvaluator):
    """
    Evaluates exact retrieval over the 10k CT26 collection.
    Returns metrics for SentenceTransformerTrainer best-checkpoint selection.
    """

    def __init__(
        self,
        doc_ids_list,
        doc_texts_list,
        docid_to_idx,
        dev_by_lang,
        doc_batch_size=8,
        query_batch_size=16,
        name="ct26-dev",
    ):
        super().__init__()
        self.doc_ids_list = doc_ids_list
        self.doc_texts_list = doc_texts_list
        self.docid_to_idx = docid_to_idx
        self.dev_by_lang = dev_by_lang
        self.doc_batch_size = doc_batch_size
        self.query_batch_size = query_batch_size
        self.name = name

        self.primary_metric = "multilingual_avg_mrr@5"
        self.greater_is_better = True

    def __call__(self, model, output_path=None, epoch=-1, steps=-1):
        model.eval()

        print(f"\n===== CT26 DEV EVAL | epoch={epoch}, steps={steps} =====")

        with torch.inference_mode():
            doc_matrix = model.encode(
                self.doc_texts_list,
                batch_size=self.doc_batch_size,
                convert_to_numpy=True,
                normalize_embeddings=True,
                show_progress_bar=True,
            ).astype(np.float32)

        per_lang_scores = {}

        for lang in LANGUAGES:
            payload = self.dev_by_lang[lang]

            with torch.inference_mode():
                q_matrix = model.encode(
                    payload["queries"],
                    batch_size=self.query_batch_size,
                    convert_to_numpy=True,
                    normalize_embeddings=True,
                    show_progress_bar=True,
                ).astype(np.float32)

            scores, _ = compute_metrics(
                q_emb=q_matrix,
                doc_emb=doc_matrix,
                gt_ids=payload["labels"],
                docid_to_idx=self.docid_to_idx,
            )

            per_lang_scores[lang] = scores

            del q_matrix
            cleanup_cuda()

        avg_scores = {}
        for metric in per_lang_scores["en"].keys():
            avg_scores[metric] = float(np.mean([per_lang_scores[lang][metric] for lang in LANGUAGES]))

        metrics = {
            "per_language": per_lang_scores,
            "average": avg_scores,
        }

        print_metrics("Qwen3-8B LoRA DEV EVAL", metrics)
        compare_to_e5(metrics, f"epoch={epoch}, steps={steps}")

        flat_metrics = {}

        for lang in LANGUAGES:
            for key, val in per_lang_scores[lang].items():
                flat_metrics[f"{lang.lower()}_{key.lower().replace('@', '@')}"] = val

        flat_metrics["multilingual_avg_mrr@1"] = avg_scores["MRR@1"]
        flat_metrics["multilingual_avg_mrr@5"] = avg_scores["MRR@5"]
        flat_metrics["multilingual_avg_mrr@10"] = avg_scores["MRR@10"]
        flat_metrics["multilingual_avg_recall@5"] = avg_scores["Recall@5"]
        flat_metrics["multilingual_avg_recall@10"] = avg_scores["Recall@10"]
        flat_metrics["multilingual_avg_recall@50"] = avg_scores["Recall@50"]
        flat_metrics["multilingual_avg_recall@100"] = avg_scores["Recall@100"]

        if output_path is not None:
            os.makedirs(output_path, exist_ok=True)
            save_json(
                os.path.join(output_path, f"ct26_dev_eval_epoch_{epoch}_steps_{steps}.json"),
                {
                    "epoch": epoch,
                    "steps": steps,
                    "metrics": metrics,
                    "flat_metrics": flat_metrics,
                },
            )

        del doc_matrix
        cleanup_cuda()

        return flat_metrics

evaluator = CT26QwenDevEvaluator(
    doc_ids_list=doc_ids_list,
    doc_texts_list=doc_texts_list,
    docid_to_idx=docid_to_idx,
    dev_by_lang=dev_by_lang,
    doc_batch_size=EVAL_DOC_BATCH_SIZE,
    query_batch_size=EVAL_QUERY_BATCH_SIZE,
)

# ==========================================
# 9. LOAD QWEN3-EMBEDDING-8B WITH FLASH ATTENTION 2
# ==========================================
print("\nLoading Qwen3-Embedding-8B with FlashAttention 2...")

model_kwargs = {
    "attn_implementation": "flash_attention_2",
    "device_map": {"": 0},
}

# Transformers 5 prefers dtype; older Transformers uses torch_dtype.
# SentenceTransformer passes model_kwargs into AutoModel.from_pretrained.
try:
    model = SentenceTransformer(
        MODEL_NAME,
        model_kwargs={
            **model_kwargs,
            "dtype": torch.bfloat16,
        },
        tokenizer_kwargs={
            "padding_side": "left",
        },
    )
except TypeError:
    model = SentenceTransformer(
        MODEL_NAME,
        model_kwargs={
            **model_kwargs,
            "torch_dtype": torch.bfloat16,
        },
        tokenizer_kwargs={
            "padding_side": "left",
        },
    )

model.max_seq_length = MAX_SEQ_LENGTH
model.tokenizer.padding_side = "left"

print("Loaded model.")
print("Max seq length:", model.max_seq_length)
print("Tokenizer padding side:", model.tokenizer.padding_side)

# ==========================================
# 10. ADD LORA ADAPTER
# ==========================================
print("\nAdding LoRA adapter...")

peft_config = LoraConfig(
    task_type=TaskType.FEATURE_EXTRACTION,
    inference_mode=False,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    target_modules=LORA_TARGET_MODULES,
)

model.add_adapter(peft_config)

# Disable cache + enable gradient checkpointing for memory.
try:
    auto_model = model[0].auto_model
    auto_model.config.use_cache = False
    auto_model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    print("Enabled gradient checkpointing on inner auto_model.")
except Exception as e:
    print("Could not manually enable gradient checkpointing:", repr(e))

try:
    model[0].auto_model.print_trainable_parameters()
except Exception:
    trainable = 0
    total = 0
    for _, p in model.named_parameters():
        total += p.numel()
        if p.requires_grad:
            trainable += p.numel()
    print(f"Trainable params: {trainable:,} / {total:,} ({100 * trainable / total:.4f}%)")

cleanup_cuda()

# ==========================================
# 11. LOSS
# ==========================================
# This keeps the real in-batch negative batch size at 128, like E5,
# while internally forwarding tiny mini-batches through Qwen3-8B.
train_loss = losses.CachedMultipleNegativesRankingLoss(
    model=model,
    mini_batch_size=CACHED_LOSS_MINI_BATCH_SIZE,
    scale=20.0,  # equivalent to temperature 0.05; standard for MNRL-style training
    show_progress_bar=False,
)

# Same warmup logic as your E5 script.
warmup_steps = int((len(train_dataset) // EFFECTIVE_BATCH_SIZE) * MAX_EPOCHS * 0.1)
print("Warmup steps:", warmup_steps)

# ==========================================
# 12. TRAINING ARGUMENTS
# ==========================================
args = SentenceTransformerTrainingArguments(
    output_dir=CHECKPOINT_DIR,

    # Keep close to E5 experiment
    num_train_epochs=MAX_EPOCHS,
    per_device_train_batch_size=EFFECTIVE_BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    warmup_steps=warmup_steps,

    # BF16 is preferred on A100
    fp16=False,
    bf16=True,

    # Important for memory
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},

    # Logging
    report_to="none",
    logging_steps=20,
    logging_first_step=True,

    # Same logic as E5: eval/save by epoch, load best
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_multilingual_avg_mrr@5",
    greater_is_better=True,

    # Important for MultipleNegativesRankingLoss
    batch_sampler=BatchSamplers.NO_DUPLICATES,

    # Safety
    dataloader_drop_last=True,
    remove_unused_columns=False,

    run_name="ct26-qwen3-embedding-8b-lora",
)

# ==========================================
# 13. TRAIN
# ==========================================
trainer = SentenceTransformerTrainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    loss=train_loss,
    evaluator=evaluator,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=EARLY_STOPPING_PATIENCE)],
)

print("\nStarting Qwen3-Embedding-8B LoRA training...")
trainer.train()

print("\nTraining complete. Best model should now be loaded because load_best_model_at_end=True.")

# ==========================================
# 14. SAVE BEST LORA MODEL
# ==========================================
print("\nSaving best LoRA SentenceTransformer model...")
model.save_pretrained(BEST_MODEL_DIR)
print("Best model saved to:", BEST_MODEL_DIR)

# Optional checkpoint cleanup
if DELETE_TRAINER_CHECKPOINTS_AFTER_EXPORT:
    print("Deleting old checkpoint folders...")
    for name in os.listdir(CHECKPOINT_DIR):
        path = os.path.join(CHECKPOINT_DIR, name)
        if os.path.isdir(path) and name.startswith("checkpoint-"):
            shutil.rmtree(path)

cleanup_cuda()

# ==========================================
# 15. FINAL DEV EVAL + SAVE ARTIFACTS
# ==========================================
print("\nEncoding full collection with BEST Qwen3 LoRA model...")

with torch.inference_mode():
    doc_matrix = model.encode(
        doc_texts_list,
        batch_size=EVAL_DOC_BATCH_SIZE,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=True,
    ).astype(np.float32)

DOC_EMB_PATH = os.path.join(ARTIFACTS_DIR, "qwen3_8b_lora_doc_embeddings.npy")
np.save(DOC_EMB_PATH, doc_matrix)
print("Saved doc embeddings to:", DOC_EMB_PATH)

all_results = {}
all_ranks = {}
dev_candidates_by_lang = {}

for lang in LANGUAGES:
    print(f"\n--- FINAL DEV Language: {lang.upper()} ---")

    payload = dev_by_lang[lang]
    rows = payload["raw_rows"]
    queries = payload["queries"]
    labels = payload["labels"]

    with torch.inference_mode():
        query_matrix = model.encode(
            queries,
            batch_size=EVAL_QUERY_BATCH_SIZE,
            convert_to_numpy=True,
            normalize_embeddings=True,
            show_progress_bar=True,
        ).astype(np.float32)

    lang_scores, ranks = compute_metrics(
        q_emb=query_matrix,
        doc_emb=doc_matrix,
        gt_ids=labels,
        docid_to_idx=docid_to_idx,
    )

    all_results[lang] = lang_scores
    all_ranks[lang] = ranks

    for k, v in lang_scores.items():
        print(f"  {k:10s}: {v:.4f}")

    if SAVE_FINAL_DEV_CANDIDATES:
        top_ids, top_scores = get_topk_candidates(
            q_emb=query_matrix,
            doc_emb=doc_matrix,
            doc_ids_list=doc_ids_list,
            topk=SAVE_CANDIDATE_TOPK,
        )

        samples = []
        for i, row in enumerate(rows):
            raw_query = normalize_ws(row["text"])
            candidate_ids = top_ids[i]
            candidate_texts = [doc_raw[cid] for cid in candidate_ids]

            samples.append({
                "query": raw_query,
                "query_with_instruction": queries[i],
                "gold_id": labels[i],
                "candidate_ids": candidate_ids,
                "candidate_texts": candidate_texts,
                "dense_scores": top_scores[i],
                "retriever_scores": top_scores[i],
                "rank_of_gold": int(ranks[i]),
            })

        dev_candidates_by_lang[lang] = samples

    del query_matrix
    cleanup_cuda()

avg_results = {}
for metric in all_results["en"].keys():
    avg_results[metric] = float(np.mean([all_results[lang][metric] for lang in LANGUAGES]))

final_metrics = {
    "per_language": all_results,
    "average": avg_results,
}

print_metrics("FINAL Qwen3-Embedding-8B LoRA DEV", final_metrics)
compare_to_e5(final_metrics, "FINAL Qwen3-8B LoRA")

FINAL_METRICS_PATH = os.path.join(EVAL_DIR, "qwen3_8b_lora_final_dev_metrics.json")
save_json(FINAL_METRICS_PATH, final_metrics)
print("Saved final metrics to:", FINAL_METRICS_PATH)

if SAVE_FINAL_DEV_CANDIDATES:
    DEV_CANDIDATES_PATH = os.path.join(
        ARTIFACTS_DIR,
        f"qwen3_8b_lora_dev_candidates_top{SAVE_CANDIDATE_TOPK}.json.gz",
    )
    save_json_gz(DEV_CANDIDATES_PATH, dev_candidates_by_lang)
    print("Saved dev candidates to:", DEV_CANDIDATES_PATH)

# ==========================================
# 16. SAVE META
# ==========================================
meta = {
    "dataset_name": DATASET_NAME,
    "languages": LANGUAGES,
    "model_name": MODEL_NAME,
    "task_instruction": TASK_INSTRUCTION,
    "root_dir": ROOT_DIR,
    "checkpoint_dir": CHECKPOINT_DIR,
    "best_model_dir": BEST_MODEL_DIR,
    "artifacts_dir": ARTIFACTS_DIR,
    "eval_dir": EVAL_DIR,

    "training": {
        "method": "LoRA",
        "loss": "CachedMultipleNegativesRankingLoss",
        "effective_batch_size": EFFECTIVE_BATCH_SIZE,
        "cached_loss_mini_batch_size": CACHED_LOSS_MINI_BATCH_SIZE,
        "max_epochs": MAX_EPOCHS,
        "learning_rate": LEARNING_RATE,
        "warmup_steps": warmup_steps,
        "early_stopping_patience": EARLY_STOPPING_PATIENCE,
        "metric_for_best_model": "eval_multilingual_avg_mrr@5",
        "batch_sampler": "NO_DUPLICATES",
    },

    "qwen": {
        "max_seq_length": MAX_SEQ_LENGTH,
        "flash_attention_2_required": True,
        "bf16": True,
        "padding_side": "left",
    },

    "lora": {
        "r": LORA_R,
        "alpha": LORA_ALPHA,
        "dropout": LORA_DROPOUT,
        "target_modules": LORA_TARGET_MODULES,
        "task_type": "FEATURE_EXTRACTION",
    },

    "files": {
        "doc_embeddings": DOC_EMB_PATH,
        "doc_ids_list": os.path.join(ARTIFACTS_DIR, "doc_ids_list.json"),
        "doc_raw": os.path.join(ARTIFACTS_DIR, "doc_raw.json.gz"),
        "final_metrics": FINAL_METRICS_PATH,
        "dev_candidates": (
            os.path.join(ARTIFACTS_DIR, f"qwen3_8b_lora_dev_candidates_top{SAVE_CANDIDATE_TOPK}.json.gz")
            if SAVE_FINAL_DEV_CANDIDATES else None
        ),
    },

    "e5_baseline": E5_BASELINE,
}

META_PATH = os.path.join(ROOT_DIR, "meta.json")
save_json(META_PATH, meta)

print("\nDone.")
print("Best model dir:", BEST_MODEL_DIR)
print("Artifacts dir:", ARTIFACTS_DIR)
print("Eval dir:", EVAL_DIR)
print("Meta:", META_PATH)

cleanup_cuda()

Mounted at /content/drive
Existing ROOT_DIR found. Deleting it:
/content/drive/MyDrive/ct26_qwen3_embedding_8b_lora
ROOT_DIR: /content/drive/MyDrive/ct26_qwen3_embedding_8b_lora

Loading scientific collection...


README.md: 0.00B [00:00, ?B/s]

collection_data.json:   0%|          | 0.00/36.5M [00:00<?, ?B/s]

Generating collection split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Preparing collection...


...

  MRR@1     : 0.6315
  MRR@5     : 0.7048
  MRR@10    : 0.7102
  MRR@50    : 0.7140
  MRR@100   : 0.7143
  Recall@1  : 0.6315
  Recall@5  : 0.8115
  Recall@10 : 0.8502
  Recall@50 : 0.9252
  Recall@100: 0.9455

--- FINAL DEV Language: FR ---


Batches:   0%|          | 0/22 [00:00<?, ?it/s]

  MRR@1     : 0.6425
  MRR@5     : 0.7101
  MRR@10    : 0.7179
  MRR@50    : 0.7201
  MRR@100   : 0.7205
  Recall@1  : 0.6425
  Recall@5  : 0.8063
  Recall@10 : 0.8632
  Recall@50 : 0.9117
  Recall@100: 0.9373

--- FINAL DEV Language: DE ---


Batches:   0%|          | 0/13 [00:00<?, ?it/s]

  MRR@1     : 0.5699
  MRR@5     : 0.6226
  MRR@10    : 0.6345
  MRR@50    : 0.6391
  MRR@100   : 0.6397
  Recall@1  : 0.5699
  Recall@5  : 0.7021
  Recall@10 : 0.7902
  Recall@50 : 0.8834
  Recall@100: 0.9249

===== FINAL Qwen3-Embedding-8B LoRA DEV =====
--- EN ---
  MRR@1     : 0.6315
  MRR@5     : 0.7048
  MRR@10    : 0.7102
  MRR@50    : 0.7140
  MRR@100   : 0.7143
  Recall@1  : 0.6315
  Recall@5  : 0.8115
  Recall@10 : 0.8502
  Recall@50 : 0.9252
  Recall@100: 0.9455
--- FR ---
  MRR@1     : 0.6425
  MRR@5     : 0.7101
  MRR@10    : 0.7179
  MRR@50    : 0.7201
  MRR@100   : 0.7205
  Recall@1  : 0.6425
  Recall@5  : 0.8063
  Recall@10 : 0.8632
  Recall@50 : 0.9117
  Recall@100: 0.9373
--- DE ---
  MRR@1     : 0.5699
  MRR@5     : 0.6226
  MRR@10    : 0.6345
  MRR@50    : 0.6391
  MRR@100   : 0.6397
  Recall@1  : 0.5699
  Recall@5  : 0.7021
  Recall@10 : 0.7902
  Recall@50 : 0.8834
  Recall@100: 0.9249
--- MULTILINGUAL AVG ---
  MRR@1     : 0.6146
  MRR@5     : 0.6792
  MRR@10    :

In [4]:
from google.colab import runtime
runtime.unassign()
